### GoEmotions Data Processing
This notebook processes the GoEmotions dataset to prepare it for emotion classification. It includes text cleaning, normalization, and filtering to remove noise and improve data quality.

The dataset originally contains multi-label emotional annotations, which are later structured for model training. This stage ensures that raw textual data is converted into a clean and usable format for emotion detection.

In [1]:
import pandas as pd
import numpy as np
import re
import joblib

In [2]:
goemotions_raw = pd.read_csv("../data/goemotions.csv")
print("Original Dataset Shape")
print(goemotions_raw.shape)
display(goemotions_raw.head())

Original Dataset Shape
(211225, 37)


,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [3]:
emotion_label_columns = goemotions_raw.columns[9:]
goemotions_filtered = goemotions_raw[["text"] + list(emotion_label_columns)]
print("Filtered Dataset Shape")
print(goemotions_filtered.shape)

Filtered Dataset Shape
(211225, 29)


In [4]:
goemotions_filtered = goemotions_filtered[goemotions_filtered[emotion_label_columns].sum(axis = 1) > 0]
print("After Removing Empty Emotion Rows")
print(goemotions_filtered.shape)

After Removing Empty Emotion Rows
(207814, 29)


In [5]:
def clean_text_function(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [6]:
goemotions_filtered["clean_text"] = goemotions_filtered["text"].apply(clean_text_function)
goemotions_filtered = goemotions_filtered[goemotions_filtered["clean_text"].str.strip() != ""]
print("After Cleaning Text")
print(goemotions_filtered.shape)
display(goemotions_filtered[["text", "clean_text"]].head())

After Cleaning Text
(207805, 30)


,text,clean_text
0,That game hurt.,that game hurt
2,"You do right, if you don't care then fuck 'em!",you do right if you don t care then fuck em
3,Man I love reddit.,man i love reddit
4,"[NAME] was nowhere near them, he was by the Fa...",name was nowhere near them he was by the falcon
5,Right? Considering it’s such an important docu...,right considering it s such an important docum...


In [7]:
goemotions_processed = goemotions_filtered.drop(columns = ["text"])
print("Final Preprocessed Dataset Shape")
print(goemotions_processed.shape)
display(goemotions_processed.head())

Final Preprocessed Dataset Shape
(207805, 29)


,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,disappointment,...,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral,clean_text
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,that game hurt
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,you do right if you don t care then fuck em
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,man i love reddit
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,name was nowhere near them he was by the falcon
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,right considering it s such an important docum...


In [8]:
joblib.dump(list(emotion_label_columns), "../models/goemotions_label_names.pkl")
goemotions_processed.to_csv("../data/goemotions_processed.csv", index = False)
print("Emotion label names saved successfully")
print("GoEmotions preprocessing completed successfully")

Emotion label names saved successfully
GoEmotions preprocessing completed successfully
